In [1]:
from pyspark.sql.functions import (
    col, trim, upper, when, current_timestamp,
    lit, to_date, round as spark_round
)
from pyspark.sql.types import DecimalType
from datetime import datetime

BRONZE_TABLE  = "Interac_Bronze.dbo.interchange_fees"
SILVER_TABLE  = "silver_interchange_fees"
SILVER_DB     = "Interac_Fabric_Workspace.Interac_Silver.dbo"
PIPELINE_NAME = "NB_08_Silver_InterchangeFees"
BATCH_DATE    = datetime.now().strftime("%Y-%m-%d")

print(f"Silver Interchange Fees Pipeline")
print(f"Started: {datetime.now()}")

StatementMeta(, 42b4d8fa-9342-4197-b278-949363a8b7e7, 3, Finished, Available, Finished, False)

Silver Interchange Fees Pipeline
Started: 2026-05-06 00:30:16.157876


In [2]:
df_bronze = spark.read.table(BRONZE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze rows read: {total_bronze:,}")

dq_results = {}
dq_results["null_rule_id"] = df_bronze.filter(col("rule_id").isNull()).count()
dq_results["null_base_rate"] = df_bronze.filter(col("base_rate_pct").isNull()).count()
dq_results["null_flat_fee"] = df_bronze.filter(col("flat_fee_cad").isNull()).count()
dq_results["zero_base_rate"] = df_bronze.filter(
    col("base_rate_pct").cast(DecimalType(10,6)) <= 0).count()
dq_results["min_fee_exceeds_max"] = df_bronze.filter(
    col("minimum_fee_cad").cast(DecimalType(10,4)) >
    col("maximum_fee_cad").cast(DecimalType(10,4))).count()
dq_results["inactive_rules"] = df_bronze.filter(col("status") != "ACTIVE").count()

print("\nDQ CHECK RESULTS:")
print("-" * 45)
for check, count_val in dq_results.items():
    status = "⚠ FLAGGED" if count_val > 0 else "✓ PASSED"
    print(f"{check:<35} {count_val:>6,}  {status}")

StatementMeta(, 42b4d8fa-9342-4197-b278-949363a8b7e7, 4, Finished, Available, Finished, False)

Bronze rows read: 168

DQ CHECK RESULTS:
---------------------------------------------
null_rule_id                             0  ✓ PASSED
null_base_rate                           0  ✓ PASSED
null_flat_fee                            0  ✓ PASSED
zero_base_rate                           0  ✓ PASSED
min_fee_exceeds_max                      0  ✓ PASSED
inactive_rules                           0  ✓ PASSED


In [3]:
df_quarantine = df_bronze.filter(
    col("rule_id").isNull() |
    col("base_rate_pct").isNull() |
    col("flat_fee_cad").isNull()
)
quarantine_count = df_quarantine.count()

if quarantine_count > 0:
    (df_quarantine
        .withColumn("_quarantine_reason", lit("NULL_PRIMARY_KEY_OR_RATE"))
        .withColumn("_quarantined_at", current_timestamp())
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_DB}.silver_interchange_fees_quarantine"))
    print(f"Quarantined: {quarantine_count:,} records")

df_valid = df_bronze.filter(
    col("rule_id").isNotNull() &
    col("base_rate_pct").isNotNull() &
    col("flat_fee_cad").isNotNull()
)

df_silver = (df_valid
    .withColumn("rule_id",                trim(col("rule_id")))
    .withColumn("merchant_category_code", upper(trim(col("merchant_category_code"))))
    .withColumn("network",                upper(trim(col("network"))))
    .withColumn("fee_tier",               upper(trim(col("fee_tier"))))
    .withColumn("currency",               upper(trim(col("currency"))))
    .withColumn("status",                 upper(trim(col("status"))))
    .withColumn("effective_date",
        to_date(col("effective_date"), "yyyy-MM-dd"))
    .withColumn("expiry_date",
        to_date(col("expiry_date"), "yyyy-MM-dd"))
    .withColumn("base_rate_pct",
        col("base_rate_pct").cast(DecimalType(10, 6)))
    .withColumn("flat_fee_cad",
        col("flat_fee_cad").cast(DecimalType(10, 4)))
    .withColumn("minimum_fee_cad",
        col("minimum_fee_cad").cast(DecimalType(10, 4)))
    .withColumn("maximum_fee_cad",
        col("maximum_fee_cad").cast(DecimalType(10, 4)))
    .withColumn("base_rate_bps",
        spark_round(col("base_rate_pct") * 10000, 2))
    .withColumn("is_active",
        when(col("status") == "ACTIVE", "Y").otherwise("N"))
    .withColumn("fee_band",
        when(col("base_rate_pct") < 0.001, "LOW")
        .when(col("base_rate_pct") < 0.002, "MEDIUM")
        .otherwise("HIGH"))
    .withColumn("_silver_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME))
    .withColumn("_batch_date",       lit(BATCH_DATE))
    .drop("_ingested_at", "_source_file", "_lakehouse")
)

print(f"Valid records: {df_silver.count():,}")

StatementMeta(, 42b4d8fa-9342-4197-b278-949363a8b7e7, 5, Finished, Available, Finished, False)

Valid records: 168


In [4]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .saveAsTable(f"{SILVER_DB}.{SILVER_TABLE}"))

spark.sql(f"OPTIMIZE {SILVER_DB}.{SILVER_TABLE} ZORDER BY (merchant_category_code, network)")

final_count = spark.read.table(f"{SILVER_DB}.{SILVER_TABLE}").count()

print("\n" + "="*60)
print("SILVER INTERCHANGE FEES SUMMARY")
print("="*60)
print(f"Bronze rows in    : {total_bronze:,}")
print(f"Quarantined       : {quarantine_count:,}")
print(f"Silver rows out   : {final_count:,}")
print(f"Pass rate         : {round(final_count/total_bronze*100, 2)}%")
print(f"Table             : {SILVER_DB}.{SILVER_TABLE}")
print(f"Completed at      : {datetime.now()}")
print("="*60)

StatementMeta(, 42b4d8fa-9342-4197-b278-949363a8b7e7, 6, Finished, Available, Finished, False)


SILVER INTERCHANGE FEES SUMMARY
Bronze rows in    : 168
Quarantined       : 0
Silver rows out   : 168
Pass rate         : 100.0%
Table             : Interac_Fabric_Workspace.Interac_Silver.dbo.silver_interchange_fees
Completed at      : 2026-05-06 00:31:02.207585


In [6]:
# Final Silver verification
print("="*65)
print("COMPLETE SILVER LAYER VERIFICATION")
print("="*65)

SILVER_DB = "Interac_Fabric_Workspace.Interac_Silver.dbo"

tables = [
    "silver_transactions",
    "silver_merchants",
    "silver_cardholders",
    "silver_settlements",
    "silver_disputes",
    "silver_fraud_labels",
    "silver_interchange_fees"
]

total = 0
for table in tables:
    count = spark.read.table(f"{SILVER_DB}.{table}").count()
    total += count
    print(f"{table:<35} {count:>10,} rows")

print("="*65)
print(f"{'TOTAL':<35} {total:>10,} rows")
print("="*65)

StatementMeta(, 42b4d8fa-9342-4197-b278-949363a8b7e7, 8, Finished, Available, Finished, False)

COMPLETE SILVER LAYER VERIFICATION
silver_transactions                    145,946 rows
silver_merchants                         2,000 rows
silver_cardholders                       5,000 rows
silver_settlements                      34,561 rows
silver_disputes                          3,500 rows
silver_fraud_labels                      2,800 rows
silver_interchange_fees                    168 rows
TOTAL                                  193,975 rows
